In [8]:
%connections hudi-connection
%glue_version 3.0
%region us-east-1
%worker_type G.1X
%number_of_workers 3
%additional_python_modules Faker==4.1.2
%spark_conf spark.serializer=org.apache.spark.serializer.KryoSerializer

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.4 
Connections to be included:
hudi-connection
Setting Glue version to: 3.0
Previous region: us-east-1
Setting new region to: us-east-1
Region is set to: us-east-1
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 3
Additional python modules to be included:
Faker==4.1.2
Previous Spark configuration: None
Setting new Spark configuration to: spark.serializer=org.apache.spark.serializer.KryoSerializer


In [10]:
%stop_session

There is no current session.


In [1]:
import sys
from pyspark.context import SparkContext
from pyspark.sql.session import SparkSession
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.dynamicframe import DynamicFrame
from pyspark.sql.functions import col, to_timestamp, monotonically_increasing_id, to_date, when
from pyspark.sql.functions import *
from awsglue.utils import getResolvedOptions
from pyspark.sql.types import *
from datetime import datetime
import boto3
from functools import reduce

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 3
Session ID: c912a3aa-9577-47e4-af2e-dbbf8103f8c3
Applying the following default arguments:
--glue_kernel_version 1.0.4
--enable-glue-datacatalog true
--additional-python-modules Faker==4.1.2
--conf spark.serializer=org.apache.spark.serializer.KryoSerializer
Waiting for session c912a3aa-9577-47e4-af2e-dbbf8103f8c3 to get into ready status...
Session c912a3aa-9577-47e4-af2e-dbbf8103f8c3 has been created.



In [4]:
try:
    import sys
    from pyspark.context import SparkContext
    from pyspark.sql.session import SparkSession
    from awsglue.context import GlueContext
    from awsglue.job import Job
    from awsglue.dynamicframe import DynamicFrame
    from pyspark.sql.functions import col, to_timestamp, monotonically_increasing_id, to_date, when
    from pyspark.sql.functions import *
    from awsglue.utils import getResolvedOptions
    from pyspark.sql.types import *
    from datetime import datetime, date
    import boto3
    from functools import reduce
    from pyspark.sql import Row
    import uuid
    from faker import Faker
except Exception as e:
    print("Modules are missing : {} ".format(e))

job_start_ts = datetime.now()
ts_format = '%Y-%m-%d %H:%M:%S'

In [5]:
sc = spark.sparkContext
glueContext = GlueContext(sc)
job = Job(glueContext)
logger = glueContext.get_logger()



global faker
faker = Faker()

### generate the data and create the dataframe

In [61]:
class DataGenerator(object):

    @staticmethod
    def get_data():
        return [
            (
                uuid.uuid4().__str__(),
                faker.name(),
                faker.random_element(elements=('IT', 'HR', 'Sales', 'Marketing')),
                faker.random_element(elements=('CA', 'NY', 'TX', 'FL', 'IL', 'RJ')),
                str(faker.random_int(min=10000, max=150000)),
                str(faker.random_int(min=18, max=60)),
                str(faker.random_int(min=0, max=100000)),
                str(faker.unix_time()),
                faker.email(),
                faker.credit_card_number(card_type='amex'),
                faker.date()
            ) for x in range(100)
        ]


data = DataGenerator.get_data()
columns = ["emp_id", "employee_name", "department", "state", "salary", "age", "bonus", "ts", "email", "credit_card","date"]

spark_df = spark.createDataFrame(data=data, schema=columns)
spark_df.count()

100


### load the table

In [62]:
db_name = "hudidb"
table_name = "employees"
recordkey = 'emp_id'
precombine = "ts"
PARTITION_FIELD = 'state'
path = "s3://dms-g/dms/"
method = 'bulk_insert'
table_type = "COPY_ON_WRITE"
bucket_name = 'dms-g'
file_key = 'dms/tables_json/employees.json'


hudi_part_write_config = {
    'className': 'org.apache.hudi',

    'hoodie.table.name': table_name,
    'hoodie.datasource.write.table.type': table_type,
    'hoodie.datasource.write.operation': method,
    'hoodie.bulkinsert.sort.mode': "NONE",
    'hoodie.datasource.write.recordkey.field': recordkey,
    'hoodie.datasource.write.precombine.field': precombine,

    'hoodie.datasource.hive_sync.mode': 'hms',
    'hoodie.datasource.hive_sync.enable': 'true',
    'hoodie.datasource.hive_sync.use_jdbc': 'false',
    'hoodie.datasource.hive_sync.support_timestamp': 'false',
    'hoodie.datasource.hive_sync.database': db_name,
    'hoodie.datasource.hive_sync.table': table_name,

}

spark_df.write.format("hudi").options(**hudi_part_write_config).mode("append").save(path)


### get max commit time from hudi dataset

In [44]:
def get_max_commit_time(path):
    parquet_file_s3_path = path
    hudi_commit_col_name = "_hoodie_commit_time"
    hudi_read_options = {
        'hoodie.datasource.query.type': 'snapshot',
        'hoodie.datasource.read.paths': parquet_file_s3_path
    }
    commits_source_df_reader = spark.read.format("org.apache.hudi").options(**hudi_read_options)
    commits_source_df = commits_source_df_reader.load()
    commits_distinct_df = commits_source_df.select(hudi_commit_col_name).distinct()
    commits = list(map(lambda row: row[0], commits_distinct_df.collect()))
    commits.sort(reverse=True)
    beginTime = commits[0]
    return beginTime
    

### call the get_max_commit_time fun to get the commit value

In [45]:
last_commit = get_max_commit_time(path)
last_commit

'20240215230302655'


### store the max commit value in the s3 file

In [11]:
import json
from datetime import datetime
import boto3

def put_files(Response, Key, BucketName):
    try:
        client = boto3.client("s3")
        response = client.put_object(
            Body=Response, Bucket=BucketName, Key=Key
        )
        return "ok"
    except Exception as e:
        raise Exception("Error : {} ".format(e))
        
def push_meta_data(json_data):
    file_name = "dms/tables_json/employees.json"
    BucketName = "dms-g"  # Define the BucketName here or pass it as an argument
    put_files(
        Response=json_data,
        Key=file_name,
        BucketName=BucketName
    )
		

json_data = {
    "last_processed_commit": last_commit,
    "table_name": table_name,
    "path": path,
    "inserted_time": datetime.now().__str__()
}

push_meta_data(json.dumps(json_data))


### get the start commit so full load can be done

In [12]:
def get_begin_commit(path):
        spark.read.format("hudi").load(path).createOrReplaceTempView("hudi_snapshot")
        commits = list(map(lambda row: row[0], spark.sql(
            "select distinct(_hoodie_commit_time) as commitTime from  hudi_snapshot order by commitTime asc").limit(
            50).collect()))
        begin_time = int(commits[0]) #taking the first commit time from the list
        return begin_time

In [13]:
begin_time = get_begin_commit(path)
begin_time

20240215183535379


### check if metadata file exists in s3

In [14]:
def check_file_exists(bucket_name, key):
    s3 = boto3.client('s3')
    try:
        s3.head_object(Bucket=bucket_name, Key=key)
        return True
    except Exception as e:
        if e.response['Error']['Code'] == '404':
            return False
        else:
            raise


In [16]:
exists = check_file_exists(bucket_name, file_key)
exists

True


### create s3 client

In [17]:
s3_client = boto3.client('s3')

### read metadata file from s3

In [21]:
def read_meta_data(s3_client, bucket, s3_key):
    try:
        data = fetch_data_from_s3(s3_client, bucket, s3_key)
        data = json.loads(data)
        return data
    except Exception as e:
        print("Failed to fetch json txt file into JSON: ", e)


def fetch_data_from_s3(s3_client, bucket, s3_key):
    response = s3_client.get_object(Bucket=bucket, Key=file_key)
    data = response["Body"].read()
    return data

In [22]:
meta_data = read_meta_data(s3_client, bucket_name, file_key)
meta_data

{'last_processed_commit': '20240215224112956', 'table_name': 'employees', 'path': 's3://dms-g/dms/', 'inserted_time': '2024-02-15 22:42:06.566422'}


### read incremental data
### we are calculating the commit_time and passing here explicitly so same function will work for both full and incremental load.


In [55]:
def read_full_inc_data(path, commit_time): 
        incremental_read_options = {
            'hoodie.datasource.query.type': 'incremental',
            'hoodie.datasource.read.begin.instanttime': commit_time,
        }
        incremental_df = spark.read.format("hudi").options(**incremental_read_options).load(path).createOrReplaceTempView("hudi_incremental")
        df = spark.sql("select * from  hudi_incremental")
        return df

### if it is full load, we are invoking get_begin_commit utility which will give us the first commit time (when the data got first loaded) so we are reading from the actual begining
### if it incremental then, we check if metadata file exists in s3, if exists, read the commit time from it and then retruns the df with the results. 
### if metadata file does not exists, will do the full load and create the metadata file in s3

In [57]:
def get_hudi_data(path, load_type):
    if load_type.lower() == 'full':
        print("it is full load")
        read_commit = get_begin_commit(path)
        full_df = read_full_inc_data(path, read_commit)
        return full_df
    
    else:
        print("it is incremental")
        is_file_exists = check_file_exists(bucket_name, file_key)
        if is_file_exists:
            meta_data = fetch_json_from_s3_text(s3_client, bucket_name, file_key)
            last_processed_commit=str(meta_data.get("last_processed_commit"))
            incr_df = read_full_inc_data(path, last_processed_commit)
            return incr_df
        else:
            print("metdata file does not exists in the s3. opting to create metadata file and go for full read")
            read_commit = get_begin_commit(path)
            incr_df = read_full_inc_data(path, read_commit)
            json_data = {
                "last_processed_commit": last_commit,
                "table_name": table_name,
                "path": path,
                "inserted_time": datetime.now().__str__()
                        }

            push_meta_data(json.dumps(json_data))
            return incr_df
            
            
            
            
            

In [63]:
df = get_hudi_data(path, 'incremental')
df.count()

it is incremental
100
